# Mixed Precision Comparison: FP16 vs BF16 vs FP32

## Overview

Comprehensive comparison of numerical formats for deep learning training.

### Topics Covered
- Numerical representation
- Range and precision trade-offs
- Hardware support
- When to use each format

## 1. Bit Layout Comparison

```
FP32: [1 sign][8 exponent][23 mantissa] = 32 bits
      Range: ±3.4e38, Precision: ~7 decimal digits

FP16: [1 sign][5 exponent][10 mantissa] = 16 bits
      Range: ±65504, Precision: ~3 decimal digits

BF16: [1 sign][8 exponent][7 mantissa] = 16 bits
      Range: ±3.4e38, Precision: ~2 decimal digits
```

In [ ]:
import torch

def compare_formats():
    """Compare numerical formats."""
    
    # Test values
    values = [1e-8, 1e-4, 1.0, 1e4, 1e38]
    
    print(f"{'Value':<12} {'FP32':<15} {'FP16':<15} {'BF16':<15}")
    print("-" * 57)
    
    for v in values:
        fp32 = torch.tensor(v, dtype=torch.float32)
        fp16 = fp32.half()
        bf16 = fp32.bfloat16()
        
        print(f"{v:<12.0e} {fp32.item():<15.2e} {fp16.item():<15.2e} {bf16.item():<15.2e}")

compare_formats()

## 2. Decision Matrix

| Criterion | FP32 | FP16 | BF16 |
|-----------|------|------|------|
| Memory | 4 bytes | 2 bytes | 2 bytes |
| Speed | 1x | ~2x | ~2x |
| Stability | Best | Needs scaling | Good |
| GPU Support | All | Volta+ | Ampere+ |

In [ ]:
def recommend_precision():
    """Recommend precision based on hardware."""
    
    if not torch.cuda.is_available():
        print("No GPU: Use FP32")
        return "fp32"
    
    capability = torch.cuda.get_device_capability()
    
    if capability >= (8, 0):  # Ampere+
        print(f"GPU capability {capability}: Use BF16 (best stability)")
        return "bf16"
    elif capability >= (7, 0):  # Volta/Turing
        print(f"GPU capability {capability}: Use FP16 with GradScaler")
        return "fp16"
    else:
        print(f"GPU capability {capability}: Use FP32")
        return "fp32"

recommend_precision()

## 3. Summary

### Quick Guide

- **FP32**: Debugging, baseline, old GPUs
- **FP16**: Volta/Turing GPUs, need GradScaler
- **BF16**: Ampere+ GPUs, simplest and most stable